# microGPT Notebook版

このNotebookは、Andrej Karpathyさんの `microgpt.py` を参考に、**PyTorchもNumPyも使わず、Python標準ライブラリだけでGPT風の小さな文字生成モデルを学習する**ためのものです。

ローカルのJupyter Notebook / JupyterLabで動かす想定です。`!pip install`、GPU前提の処理は入れていません。

元コードの中心思想はかなりシンプルです。

- データは文字列のリスト
- Tokenizerは文字を整数IDにするだけ
- 自動微分は `Value` というスカラー計算グラフだけで実装
- モデルはGPT風のTransformer decoder
- 最適化はAdamを手書き
- 推論は次の文字を確率的に選ぶだけ

つまり、これは「実用的に速いGPT」ではなく、**GPTの学習と推論の全体像を、最小限の部品だけで見えるようにした教材**です。

## 0. このコードが何をしているか

このNotebookで作るものは、名前データセットのような短い文字列を読み、次の文字を予測する小さな文字レベル言語モデルです。

たとえば、学習データに `emma`, `olivia`, `liam` のような名前が入っているとします。モデルは、

```text
BOS -> e
BOS, e -> m
BOS, e, m -> m
BOS, e, m, m -> a
BOS, e, m, m, a -> BOS
```

のように「今までの文字列から次の文字を当てる」練習をします。ここで `BOS` は Beginning of Sequence、つまり文字列の始まりと終わりを表す特別なトークンです。

学習後は `BOS` から始めて、1文字ずつ次の文字をサンプリングして、存在しそうで存在しない名前を生成します。

重要なのは、ここで使っているGPT風モデルが、既存ライブラリのブラックボックスではないことです。線形層、softmax、RMSNorm、Attention、MLP、loss、backward、Adam更新まで、全部このNotebook内にあります。

## 1. Jupyterで動かす前提

このNotebookはローカルJupyter向けです。

推奨する置き方は次のような形です。

```text
作業フォルダ/
  microgpt_notebook_jupyter.ipynb
  input.txt   # 任意。1行に1つの文字列を書く
```

`input.txt` がない場合は、元コードと同じく `makemore` の名前データをダウンロードしようとします。ネットワークが使えない場合でもNotebookの流れを確認できるように、ごく小さな内蔵サンプルへフォールバックします。

速度については、あえて高速化していません。全部スカラーの自動微分で動くので、普通の深層学習ライブラリよりずっと遅いです。最初は `num_steps = 50` くらいで動作確認し、問題なければ `1000` に戻すのがおすすめです。

In [1]:
import os
import math
import random
import urllib.request

random.seed(42)

## 2. データセット

`docs` は `list[str]` です。ここでは「1行に1つの名前」が入った `input.txt` を読みます。

この段階ではまだ機械学習らしいことはしていません。ただの文字列リストです。

In [2]:
INPUT_PATH = "input.txt"
NAMES_URL = "https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt"

if not os.path.exists(INPUT_PATH):
    try:
        print("input.txt がないので、名前データセットをダウンロードします...")
        urllib.request.urlretrieve(NAMES_URL, INPUT_PATH)
    except Exception as e:
        print("ダウンロードに失敗しました。Notebook確認用の小さな内蔵データを使います。")
        print("理由:", repr(e))
        fallback_docs = [
            "emma", "olivia", "ava", "isabella", "sophia",
            "liam", "noah", "oliver", "elijah", "james",
            "yui", "sora", "ren", "aoi", "hina",
        ]
        with open(INPUT_PATH, "w", encoding="utf-8") as f:
            for name in fallback_docs:
                f.write(name + "\n")

docs = [line.strip() for line in open(INPUT_PATH, encoding="utf-8") if line.strip()]
random.shuffle(docs)

print(f"num docs: {len(docs)}")
print("例:", docs[:10])

num docs: 32033
例: ['yuheng', 'diondre', 'xavien', 'jori', 'juanluis', 'erandi', 'phia', 'samatha', 'phoenix', 'emmelynn']


## 3. Tokenizer

Tokenizerと言っても、ここではBPEやSentencePieceのようなものではありません。
データに出てくるユニークな文字を集めて、それぞれに整数IDを割り当てます。

- `uchars`: データに登場した文字の一覧
- `BOS`: 始まり・終わりを表す特別トークン
- `vocab_size`: 文字の種類数 + BOS

最低限を意識するとこれになるという認識です。
言語モデルの本質は、まず「記号列の次を予測する」ことだからです。

In [3]:
uchars = sorted(set("".join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

print(f"unique chars: {''.join(uchars)}")
print(f"BOS token id: {BOS}")
print(f"vocab size: {vocab_size}")

def encode(s):
    """文字列を token id の列へ変換する。"""
    return [uchars.index(ch) for ch in s]


def decode(tokens):
    """token id の列を文字列へ戻す。BOSは表示しない。"""
    return "".join(uchars[t] for t in tokens if t != BOS)

print("encode('emma'):", encode("emma") if all(ch in uchars for ch in "emma") else "データに含まれない文字があります")

unique chars: abcdefghijklmnopqrstuvwxyz
BOS token id: 26
vocab size: 27
encode('emma'): [4, 12, 12, 0]


## 4. スカラー自動微分 `Value`

ここがこのコードの心臓部です。

普通の深層学習では、テンソル演算と自動微分をPyTorchなどに任せます。しかしこのNotebookでは、1つの数値を `Value` として持ち、その計算履歴をグラフとして保存します。

たとえば、

```python
z = x * y + x
```

と計算すると、`z` は「自分が `x*y` と `x` から作られた」ことを覚えます。最後に `loss.backward()` を呼ぶと、計算グラフを逆向きにたどって、各パラメータの勾配 `grad` を計算します。

これはmicrograd的な考え方です。テンソルではなくスカラーなので遅いですが、仕組みは非常に見えやすくなります。

In [4]:
class Value:
    __slots__ = ("data", "grad", "_children", "_local_grads")

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        # 加算: f(a, b) = a + b → df/da = 1, df/db = 1
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        # 乗算: f(a, b) = a * b → df/da = b, df/db = a
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other):
        return Value(self.data ** other, (self,), (other * self.data ** (other - 1),))

    def log(self):
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def exp(self):
        return Value(math.exp(self.data), (self,), (math.exp(self.data),))

    def relu(self):
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1

        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

### 4.1 自動微分の小さな確認

本体に入る前に、`Value` が本当に勾配を計算しているか確認します。

ここでは、

\[
z = x^2 + 3x
\]

を考えます。微分すると、

\[
\frac{dz}{dx} = 2x + 3
\]

なので、`x = 2` なら勾配は `7` になるはずです。

In [5]:
x = Value(2.0)
z = x * x + 3 * x
z.backward()

print("z.data:", z.data)
print("x.grad:", x.grad, "<- 7.0 に近ければOK")

z.data: 10.0
x.grad: 7.0 <- 7.0 に近ければOK


## 5. パラメータの初期化

ここからGPT風モデルのパラメータを作ります。

このモデルはかなり小さいです。

- `n_layer = 1`: Transformer blockは1層
- `n_embd = 16`: 埋め込み次元は16
- `block_size = 16`: 最大コンテキスト長は16文字
- `n_head = 4`: Attention headは4個

実用モデルでは桁違いに大きくなりますが、ここでは「構造が見えること」を優先します。

In [6]:
n_layer = 1    # レイヤー数（GPT-2は12〜48層）
n_embd = 16    # 埋め込み次元
block_size = 16    # 最大シーケンス長
n_head = 4    # アテンションヘッド数
head_dim = n_embd // n_head    # = 4（各ヘッドの次元）

def matrix(nout, nin, std=0.08):
    """nout x nin の重み行列を Value で作る。"""
    return [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]


state_dict = {
    "wte": matrix(vocab_size, n_embd),      # token embedding
    "wpe": matrix(block_size, n_embd),     # position embedding
    "lm_head": matrix(vocab_size, n_embd), # logitsへの出力行列
}

for i in range(n_layer):
    state_dict[f"layer{i}.attn_wq"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wk"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wv"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wo"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.mlp_fc1"] = matrix(4 * n_embd, n_embd)
    state_dict[f"layer{i}.mlp_fc2"] = matrix(n_embd, 4 * n_embd)

params = [p for mat in state_dict.values() for row in mat for p in row]

print(f"num params: {len(params)}")

num params: 4192


## 6. モデルを構成する部品

次の部品を手書きします。

- `linear`: 行列とベクトルの積
- `softmax`: logitsを確率に変換
- `rmsnorm`: 平均二乗で正規化する簡易LayerNorm系の処理
- `gpt`: 1 tokenを入力し、次tokenのlogitsを出す本体

`gpt` は1回呼ぶと「現在位置の1文字」だけを処理します。ただし `keys` と `values` に過去の情報を溜めるので、Attentionは過去の文字を見ることができます。

つまり、推論時のKV cacheのとても小さな版になっています。

In [7]:
def linear(x, w):
    """y = W x。wは [nout][nin]、xは [nin]。"""
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]


def softmax(logits):
    """Valueのlistを確率分布へ変換する。maxを引いて少しだけ数値安定化する。"""
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]


def rmsnorm(x):
    """RMSNorm。平均二乗でスケールをそろえる。"""
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]


def gpt(token_id, pos_id, keys, values):
    """1 tokenぶんのforward。戻り値は次tokenに対するlogits。"""
    tok_emb = state_dict["wte"][token_id]
    pos_emb = state_dict["wpe"][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]

    # 最初にもRMSNormを置く。
    # 残差接続を通る勾配の都合で、単なる飾りではない。
    x = rmsnorm(x)

    for li in range(n_layer):
        # 1) Multi-head self-attention block
        x_residual = x
        x = rmsnorm(x)

        q = linear(x, state_dict[f"layer{li}.attn_wq"])
        k = linear(x, state_dict[f"layer{li}.attn_wk"])
        v = linear(x, state_dict[f"layer{li}.attn_wv"])

        keys[li].append(k)
        values[li].append(v)

        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs + head_dim]
            k_h = [ki[hs:hs + head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs + head_dim] for vi in values[li]]

            attn_logits = [
                sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim ** 0.5
                for t in range(len(k_h))
            ]
            attn_weights = softmax(attn_logits)

            head_out = [
                sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h)))
                for j in range(head_dim)
            ]
            x_attn.extend(head_out)

        x = linear(x_attn, state_dict[f"layer{li}.attn_wo"])
        x = [a + b for a, b in zip(x, x_residual)]

        # 2) MLP block
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f"layer{li}.mlp_fc1"])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f"layer{li}.mlp_fc2"])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict["lm_head"])
    return logits

## 7. 1文書ぶんのloss

1つの文字列を取り出し、前から順番に次のtokenを予測します。

`loss_t = -log(prob[target_id])` なので、正解tokenに高い確率を出せるほどlossは小さくなります。

In [8]:
def loss_for_doc(doc):
    """1つのdocに対する平均negative log likelihoodを返す。"""
    tokens = [BOS] + encode(doc) + [BOS]
    n = min(block_size, len(tokens) - 1)

    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]
    losses = []

    for pos_id in range(n):
        token_id = tokens[pos_id]
        target_id = tokens[pos_id + 1]

        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)

    loss = (1 / n) * sum(losses)
    return loss

# まず1件だけforwardして、lossの形を確認します。
test_loss = loss_for_doc(docs[0])
print("doc:", docs[0])
print("loss:", test_loss.data)

doc: yuheng
loss: 3.3659669475848504


## 8. Adam optimizer

Adamも手書きします。

ここでやっていることは、各パラメータ `p` について、

- 勾配の移動平均 `m`
- 勾配二乗の移動平均 `v`
- bias correction
- 学習率に基づく更新

を計算することです。

深層学習ライブラリなら `optimizer.step()` の一言ですが、ここではその中身を見える場所に置きます。

In [9]:
learning_rate = 0.01
beta1 = 0.85
beta2 = 0.99
eps_adam = 1e-8

m = [0.0] * len(params)
v = [0.0] * len(params)

## 9. 学習ループ

これが訓練本体です。

元コードの思想に合わせて、ミニバッチもDataLoaderもありません。1ステップで1つの文字列を取り出し、その文字列上の次文字予測lossを下げます。

In [10]:
num_steps = 1000  # 動作確認だけなら 50 などに下げる

for step in range(num_steps):
    doc = docs[step % len(docs)]

    # forward
    loss = loss_for_doc(doc)

    # backward
    loss.backward()

    # Adam update with linear learning rate decay
    lr_t = learning_rate * (1 - step / num_steps)

    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2

        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))

        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    if (step + 1) % 10 == 0 or step == 0:
        print(f"step {step + 1:4d} / {num_steps:4d} | loss {loss.data:.4f}", end="\r")

print(f"\nfinal loss: {loss.data:.4f}")

step 1000 / 1000 | loss 2.6497
final loss: 2.6497


## 10. 推論: 文字を1つずつ生成する

学習後は、`BOS` から始めて次の文字をサンプリングします。
`temperature` を下げると無難な出力になり、上げると変な出力が増えます。

In [11]:
def sample_one(temperature=0.5):
    keys = [[] for _ in range(n_layer)]
    values = [[] for _ in range(n_layer)]

    token_id = BOS
    sample = []

    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])

        token_id = random.choices(
            range(vocab_size),
            weights=[p.data for p in probs],
        )[0]

        if token_id == BOS:
            break

        sample.append(uchars[token_id])

    return "".join(sample)


print("--- inference: new, hallucinated names ---")
for sample_idx in range(20):
    print(f"sample {sample_idx + 1:2d}: {sample_one(temperature=0.5)}")

--- inference: new, hallucinated names ---
sample  1: kamon
sample  2: ann
sample  3: karai
sample  4: jaire
sample  5: vialan
sample  6: karia
sample  7: yeran
sample  8: anna
sample  9: areli
sample 10: kaina
sample 11: konna
sample 12: keylen
sample 13: liole
sample 14: alerin
sample 15: earan
sample 16: lenne
sample 17: kana
sample 18: lara
sample 19: alela
sample 20: anton


## 11. 何を変えると何が起きるか

このNotebookで触ると理解が深まりやすい場所です。

### `num_steps`

学習回数です。増やすと学習が進みますが、スカラー自動微分なので遅くなります。

### `n_embd`

モデルの幅です。大きくすると表現力は上がりますが、パラメータ数と計算量が増えます。

### `n_layer`

Transformer blockの数です。増やすと深くなります。ただし、この実装ではかなり遅くなります。

### `block_size`

最大で何文字前まで見るかです。名前データなら16程度で十分です。

### `temperature`

生成時のランダムさです。

- 低い: ありがちな文字列になりやすい
- 高い: 多様だが壊れやすい
